# Routing-NLU LLM Fine-Tuning Notebook

This notebook implements and fine-tunes a **small 0.5B parameter LLM** (Qwen2.5-0.5B-Instruct)
to act as a **Routing-NLU module**:

- **Input:** messy natural language like  
  `i want to go cinema with only 56 egp and go to pharmcy on my way`  
- **Output:** a **structured JSON object** describing intent, stops, and constraints, e.g.:

```json
{
  "intent": "plan_multistop_trip",
  "main_request": "Go to the cinema and then stop at a pharmacy to get medicine",
  "stops": [
    { "type": "destination", "place": "cinema" },
    { "type": "stop_on_the_way", "place": "pharmacy" }
  ],
  "constraints": {
    "budget": "56 EGP"
  }
}
```

The notebook is documented in a style that mixes:

- **Educational depth (Option A)** – explaining important ML / LLM concepts.
- **Professional engineering clarity (Option B)** – focusing on design decisions, data formats, and reproducibility.

You can run the notebook **cell by cell in Google Colab** to reproduce the model.

## Cell 1 – Install Dependencies

We install only the essential libraries:

- `transformers` – Hugging Face library for loading, training, and using LLMs.
- `datasets` – for holding and processing our training examples.

This keeps the environment lightweight and stable, and avoids extra tools that previously caused issues.

In [1]:
!pip install -q "transformers>=4.43.0" "datasets>=2.20.0"

## Cell 2 – Import Libraries and Load Base Model

**What this cell does:**
- Imports the core classes from `transformers` and `datasets`.
- Loads the **Qwen/Qwen2.5-0.5B-Instruct** model and tokenizer.
- Ensures that the tokenizer has a valid `pad_token`, which is needed for batching and training.

**Key concepts:**

- **Tokenizer**: Converts text to token IDs and back. It also handles the chat template for Qwen.
- **Causal LM (AutoModelForCausalLM)**: A left-to-right language model which predicts the next token given previous ones. This is the standard architecture used for LLMs.
- **`device_map="auto"`**: Automatically places the model on GPU if available (e.g., in Colab), otherwise CPU.
- **Padding token**: When training in batches, all sequences need the same length. We use the EOS token as the pad token for simplicity, which is common practice in causal language models.

In [2]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset
import torch
import json

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

# Load base model (float32, safe for training)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

# Ensure pad token exists for batching
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print("Model dtype:", next(model.parameters()).dtype)
print("Pad token id:", tokenizer.pad_token_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model dtype: torch.float32
Pad token id: 151643


## Cell 3 – Define System Instruction and Training Examples

**What this cell does:**
- Defines a **global system instruction** describing the Routing-NLU behavior.
- Defines a **small supervised dataset** of `(user → JSON)` pairs.

Each training example has:
- `user`: the natural language message.
- `assistant`: the target JSON that we want the model to produce.

**Concepts:**

- This is **Supervised Fine-Tuning (SFT)**: we show the model both the input and the *desired* output.
- The JSON schema encodes your routing logic: `intent`, `main_request`, `stops`, `constraints`.
- The dataset is intentionally small and focused: we train a specialized model, not a general chatbot.

In [3]:
SYSTEM_INSTRUCTION = (
    "You are a routing-NLU module.\n"
    "Given a user message, extract only the information needed for routing "
    "and output ONLY valid JSON.\n"
    "Use this schema exactly:\n"
    "{\n"
    '  \"intent\": \"find_route | plan_multistop_trip | update_preferences | ask_status | chitchat\",\n'
    '  \"main_request\": string,\n'
    '  \"stops\": [ { \"type\": \"origin | destination | stop_on_the_way | via | home | work\", \"place\": string } ],\n'
    '  \"constraints\": { optional_keys_like_budget_or_preference }\n'
    "}\n"
    "Do not invent information. Use only what the user says.\n"
    "Output only JSON, no explanations."
)

training_data = [
    {
        "user": "i want to go cinema with only 56 egp and go to pharmcy on my way to get some medicine",
        "assistant": {
            "intent": "plan_multistop_trip",
            "main_request": "Go to the cinema and then stop at a pharmacy to get medicine",
            "stops": [
                {"type": "destination", "place": "cinema"},
                {"type": "stop_on_the_way", "place": "pharmacy"}
            ],
            "constraints": {
                "budget": "56 EGP"
            }
        }
    },
    {
        "user": "I want to go to San Stefano then continue to the library but avoid crowded buses",
        "assistant": {
            "intent": "plan_multistop_trip",
            "main_request": "Go to San Stefano then continue to the library while avoiding crowded buses",
            "stops": [
                {"type": "destination", "place": "San Stefano"},
                {"type": "stop_on_the_way", "place": "library"}
            ],
            "constraints": {
                "avoid_modes": ["crowded buses"]
            }
        }
    },
    {
        "user": "take me from ibrahimya to faculty of engineering and choose a route under 30 egp",
        "assistant": {
            "intent": "find_route",
            "main_request": "Find a route from Ibrahimya to Faculty of Engineering under 30 EGP",
            "stops": [
                {"type": "origin", "place": "ibrahimya"},
                {"type": "destination", "place": "faculty of engineering"}
            ],
            "constraints": {
                "budget": "30 EGP"
            }
        }
    },
    {
        "user": "I need to pass by the post office before reaching my office",
        "assistant": {
            "intent": "plan_multistop_trip",
            "main_request": "Pass by the post office before reaching the office",
            "stops": [
                {"type": "stop_on_the_way", "place": "post office"},
                {"type": "destination", "place": "office"}
            ],
            "constraints": {}
        }
    },
    {
        "user": "go to the beach but only if it's the shortest route available",
        "assistant": {
            "intent": "find_route",
            "main_request": "Find the shortest route to the beach",
            "stops": [
                {"type": "destination", "place": "beach"}
            ],
            "constraints": {
                "preference": "shortest"
            }
        }
    },
    {
        "user": "I want to reach downtown but avoid any metro lines",
        "assistant": {
            "intent": "find_route",
            "main_request": "Find a route to downtown avoiding metro lines",
            "stops": [
                {"type": "destination", "place": "downtown"}
            ],
            "constraints": {
                "avoid_modes": ["metro"]
            }
        }
    },
    {
        "user": "go from home to work with the least number of stops",
        "assistant": {
            "intent": "find_route",
            "main_request": "Find a route from home to work with the least number of stops",
            "stops": [
                {"type": "origin", "place": "home"},
                {"type": "destination", "place": "work"}
            ],
            "constraints": {
                "preference": "least_stops"
            }
        }
    },
    {
        "user": "take me to Alexandria stadium but I want to pass by my friend's place first",
        "assistant": {
            "intent": "plan_multistop_trip",
            "main_request": "Go to Alexandria Stadium with a stop at a friend's place first",
            "stops": [
                {"type": "stop_on_the_way", "place": "friend's place"},
                {"type": "destination", "place": "Alexandria stadium"}
            ],
            "constraints": {}
        }
    },
    {
        "user": "can you find me a cheap route to Carrefour City Center from mandara",
        "assistant": {
            "intent": "find_route",
            "main_request": "Find a cheap route to Carrefour City Center from Mandara",
            "stops": [
                {"type": "origin", "place": "mandara"},
                {"type": "destination", "place": "Carrefour City Center"}
            ],
            "constraints": {
                "preference": "cheapest"
            }
        }
    },
    {
        "user": "I need to return home but avoid passing through rush hour traffic",
        "assistant": {
            "intent": "find_route",
            "main_request": "Find a route home avoiding rush hour traffic",
            "stops": [
                {"type": "destination", "place": "home"}
            ],
            "constraints": {
                "avoid": "rush hour"
            }
        }
    },
    {
        "user": "before going to camp caesar I must stop at the vet clinic",
        "assistant": {
            "intent": "plan_multistop_trip",
            "main_request": "Stop at the vet clinic before going to Camp Caesar",
            "stops": [
                {"type": "stop_on_the_way", "place": "vet clinic"},
                {"type": "destination", "place": "Camp Caesar"}
            ],
            "constraints": {}
        }
    },
    {
        "user": "I want to go to City Center but stop at the gas station first",
        "assistant": {
            "intent": "plan_multistop_trip",
            "main_request": "Go to City Center with a stop at the gas station first",
            "stops": [
                {"type": "stop_on_the_way", "place": "gas station"},
                {"type": "destination", "place": "City Center"}
            ],
            "constraints": {}
        }
    },
    {
        "user": "take me to the train station from miami but keep the walk as short as possible",
        "assistant": {
            "intent": "find_route",
            "main_request": "Find a route from Miami to the train station with minimal walking",
            "stops": [
                {"type": "origin", "place": "miami"},
                {"type": "destination", "place": "train station"}
            ],
            "constraints": {
                "preference": "minimal_walk"
            }
        }
    },
    {
        "user": "I need to go to hyper one but avoid trams today",
        "assistant": {
            "intent": "find_route",
            "main_request": "Go to Hyper One avoiding trams",
            "stops": [
                {"type": "destination", "place": "hyper one"}
            ],
            "constraints": {
                "avoid_modes": ["tram"]
            }
        }
    },
    {
        "user": "go from my house to my sister's school then to work and keep it under 40 pounds",
        "assistant": {
            "intent": "plan_multistop_trip",
            "main_request": "Go from home to sister's school and then to work under 40 EGP",
            "stops": [
                {"type": "home", "place": "home"},
                {"type": "stop_on_the_way", "place": "sister's school"},
                {"type": "destination", "place": "work"}
            ],
            "constraints": {
                "budget": "40 EGP"
            }
        }
    },
    {
        "user": "just checking in bro, hope you're doing well",
        "assistant": {
            "intent": "chitchat",
            "main_request": "The user is greeting casually",
            "stops": [],
            "constraints": {}
        }
    },
    {
        "user": "I want to reach el shatby hospital but avoid highways",
        "assistant": {
            "intent": "find_route",
            "main_request": "Find a route to El Shatby Hospital avoiding highways",
            "stops": [
                {"type": "destination", "place": "el shatby hospital"}
            ],
            "constraints": {
                "avoid": "highways"
            }
        }
    },
    {
        "user": "take me to the gym after passing by the tailor",
        "assistant": {
            "intent": "plan_multistop_trip",
            "main_request": "Go to the gym after passing by the tailor",
            "stops": [
                {"type": "stop_on_the_way", "place": "tailor"},
                {"type": "destination", "place": "gym"}
            ],
            "constraints": {}
        }
    },
    {
        "user": "from seyouf to the corniche but take the fastest route",
        "assistant": {
            "intent": "find_route",
            "main_request": "Find the fastest route from Seyouf to the Corniche",
            "stops": [
                {"type": "origin", "place": "seyouf"},
                {"type": "destination", "place": "corniche"}
            ],
            "constraints": {
                "preference": "fastest"
            }
        }
    },
    {
        "user": "I must go to the bank before 3 pm from my office",
        "assistant": {
            "intent": "find_route",
            "main_request": "Find a route from the office to the bank before 3 PM",
            "stops": [
                {"type": "origin", "place": "office"},
                {"type": "destination", "place": "bank"}
            ],
            "constraints": {
                "time_window": "before 3 pm"
            }
        }
    },
    {
        "user": "go home but only using buses, no taxis",
        "assistant": {
            "intent": "find_route",
            "main_request": "Find a bus-only route to home",
            "stops": [
                {"type": "destination", "place": "home"}
            ],
            "constraints": {
                "allowed_modes": ["bus"]
            }
        }
    },
]

len(training_data)

21

## Cell 4 – Build Hugging Face Dataset from Training Pairs

**What this cell does:**
- Converts the list of Python dicts into a `datasets.Dataset` object.
- Constructs the **training prompts** by concatenating:
  - The global system instruction
  - The user message
  - A fixed marker `"Assistant JSON:"` indicating where the model should start generating JSON.
- Keeps the JSON target as a string in `response`.

**Concepts:**

- In causal LM training, we build one long text sequence where the model is trained to **predict the next tokens**.
- By formatting prompt + expected JSON together, we teach the model:  
  “Given this instruction and this user message, the next tokens should be this JSON.”

In [4]:
training_texts = []
targets = []

for item in training_data:
    user_msg = item["user"]
    assistant_json = json.dumps(item["assistant"], ensure_ascii=False)

    prompt = (
        SYSTEM_INSTRUCTION
        + "\n\nUser message:\n"
        + user_msg
        + "\n\nAssistant JSON:\n"
    )

    training_texts.append(prompt)
    targets.append(assistant_json)

dataset = Dataset.from_dict({
    "prompt": training_texts,
    "response": targets
})

dataset

Dataset({
    features: ['prompt', 'response'],
    num_rows: 21
})

## Cell 5 – Build Training Text and Tokenize

**What this cell does:**
1. Creates a new field `text = prompt + response` for each example.
2. Tokenizes this text into `input_ids` using the Qwen tokenizer.
3. Sets `labels = input_ids`, which is standard for **causal language modeling**.

**Concepts:**

- For a causal LM, the loss is computed as the cross-entropy between the predicted next token and the **actual next token** in the sequence.
- Here, the model sees:  
  `SYSTEM_INSTRUCTION + "User message..." + "Assistant JSON:" + <target JSON>`  
  and learns to generate the JSON portion when given only the prompt at inference time.
- `max_length` is set to 512, which is plenty for these short messages + JSON outputs.

In [5]:
max_length = 512  # enough for prompts + JSON

def build_training_text(example):
    return {
        "text": example["prompt"] + example["response"]
    }

dataset_with_text = dataset.map(build_training_text)

def tokenize_function(batch):
    tokenized = tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length,
    )
    # For causal LM training, labels are just the input_ids
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset_with_text.map(
    tokenize_function,
    batched=True,
    remove_columns=["prompt", "response", "text"],
)

tokenized_dataset

Map:   0%|          | 0/21 [00:00<?, ? examples/s]

Map:   0%|          | 0/21 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 21
})

## Cell 6 – Define Data Collator, TrainingArguments, and Train the Model

**What this cell does:**
- Uses `DataCollatorForLanguageModeling` with `mlm=False` for causal LM (not masked LM).
- Configures `TrainingArguments`:
  - `num_train_epochs=8` – multiple passes over the small dataset.
  - `per_device_train_batch_size=2` and `gradient_accumulation_steps=4` → effective batch size 8.
  - `learning_rate=2e-4` – reasonable LR for small fine-tune.
  - `fp16=False`, `bf16=False` – avoids mixed-precision issues seen earlier.
- Creates a `Trainer` and runs `.train()`.

**Concepts:**

- **Trainer**: High-level API that handles batching, loss computation, backpropagation, and checkpointing.
- **Fine-tuning**: We adjust the existing model weights slightly to specialize the model for Routing-NLU, instead of training from scratch.
- The training loss should **decrease** over epochs, indicating the model is fitting the examples.

In [6]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

output_dir = "qwen-routing-0.5b-finetuned"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=8,           # small dataset => several epochs
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=1,
    save_strategy="epoch",
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
1,3.003500
2,4.354100
3,2.822900
4,1.875000
5,0.931100
6,0.840300
7,0.361300
8,0.315300
9,0.174100
10,0.123100


TrainOutput(global_step=24, training_loss=0.6512889427443346, metrics={'train_runtime': 1873.1814, 'train_samples_per_second': 0.09, 'train_steps_per_second': 0.013, 'total_flos': 184709784010752.0, 'train_loss': 0.6512889427443346, 'epoch': 8.0})

## Cell 7 – Inference Helper: JSON Extraction and `routing_once`

**What this cell does:**
- Defines `extract_first_json_object(text)`:
  - Scans the model's raw output and extracts the first substring that is valid JSON.
  - This makes the system robust even if the model generates extra tokens after the JSON.
- Defines `routing_once(user_message)`:
  - Builds a chat prompt with a **strict system message** that demands JSON output.
  - Uses the tokenizer's `apply_chat_template` for Qwen chat formatting.
  - Calls `model.generate()` deterministically (`do_sample=False`) to get stable outputs.
  - Extracts and returns the JSON string.

Then we test it on two example messages:

1. Cinema + pharmacy + budget.  
2. Smouha → Mahatet El Raml with cheapest option.

**Concepts:**

- **Prompt engineering**: The system prompt tells the model exactly which keys to output and forbids explanations.
- **Autoregressive generation**: The model generates tokens one-by-one after the prompt, and we decode only the *new* tokens.
- **Post-processing for safety**: JSON extraction ensures your downstream routing system always receives valid JSON, not free-form text.

In [7]:
def extract_first_json_object(text: str) -> str:
    """Extract the first valid top-level JSON object from a string."""
    start = text.find("{")
    if start == -1:
        raise ValueError("No '{' found in text")

    for end in range(len(text), start, -1):
        candidate = text[start:end]
        try:
            json.loads(candidate)
            return candidate
        except json.JSONDecodeError:
            continue

    raise ValueError("Could not find a valid JSON object in text")


def routing_once(user_message: str, max_new_tokens: int = 128):
    system_prompt = (
        "You are a routing-NLU module.\n"
        "Given a user message, you MUST output ONLY one valid JSON object.\n"
        "The JSON must have exactly these keys: intent, main_request, stops, constraints.\n"
        "intent must be one of: find_route, plan_multistop_trip, update_preferences, ask_status, chitchat.\n"
        "stops must be a list of objects with keys: type and place.\n"
        "constraints must be an object (use {} if none).\n"
        "Do not add explanations. Do not add any text before or after the JSON.\n"
        "Stop immediately after finishing the JSON object."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    # Use Qwen's chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][input_length:]
    raw_output = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    json_str = extract_first_json_object(raw_output)
    return json_str


# Test 1
msg1 = "i want to go cinema with only 56 egp and go to pharmcy on my way to get some medicine"
out1 = routing_once(msg1)
print("CLEAN JSON 1:", out1)
print("PARSED 1:", json.loads(out1))

# Test 2
msg2 = "good morning bro, I want to go from smouha to mahatet el raml using the cheapest option"
out2 = routing_once(msg2)
print("\nCLEAN JSON 2:", out2)
print("PARSED 2:", json.loads(out2))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


CLEAN JSON 1: {"intent": "plan_multistop_trip", "main_request": "Go to the cinema and then stop at a pharmacy to get medicine", "stops": [{"type": "destination", "place": "cinema"}, {"type": "stop_on_the_way", "place": "pharmacy"}], "constraints": {"budget": "56 EGP"}}
PARSED 1: {'intent': 'plan_multistop_trip', 'main_request': 'Go to the cinema and then stop at a pharmacy to get medicine', 'stops': [{'type': 'destination', 'place': 'cinema'}, {'type': 'stop_on_the_way', 'place': 'pharmacy'}], 'constraints': {'budget': '56 EGP'}}

CLEAN JSON 2: {"intent": "plan_multistop_trip", "main_request": "Go from Smouf to Mahat et El Raml using the cheapest option", "stops": [{"type": "origin | destination | stop_on_the_way | via | home | work", "place": "mahatel raml"}], "constraints": {"preference": "cheapest"}}
PARSED 2: {'intent': 'plan_multistop_trip', 'main_request': 'Go from Smouf to Mahat et El Raml using the cheapest option', 'stops': [{'type': 'origin | destination | stop_on_the_way | v

## Cell 8 – Save and Export the Fine-Tuned Model

**What this cell does:**
- Saves the fine-tuned model and tokenizer to a local folder inside the Colab environment.
- Optionally compresses the folder into a `.zip` file and triggers a download to your machine.

**Concepts:**

- A Hugging Face model folder typically contains:
  - `config.json` – model architecture and config.
  - `pytorch_model.bin` – model weights.
  - Tokenizer files – vocab, merges, tokenizer config.
- Once downloaded, you can load this model elsewhere using `AutoModelForCausalLM.from_pretrained(path)` and reuse `routing_once` with the same logic.

This turns your experiment into a **reusable component** for your routing system.

In [8]:
save_dir = "qwen-routing-0.5b-finetuned-local"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print("Model and tokenizer saved to:", save_dir)

Model and tokenizer saved to: qwen-routing-0.5b-finetuned-local
